# Preliminary: NumPy Operations for KMeans

This notebook covers every NumPy operation you will encounter in the Week 3 clustering notebooks. Work through this **before** moving on to `week_3.ipynb` and `week_3_kmean.ipynb`.

In [15]:
import numpy as np

---
## 1. Creating Arrays

`np.array()` creates an array from a Python list.

```
np.array([1, 2, 3])          -> 1D array (a single data point with 3 features)
np.array([[1,2], [3,4]])     -> 2D array (2 data points, each with 2 features)
```

In KMeans, our data matrix `X` has shape `(N, d)` -- N points, d features.

In [16]:
# 1D array -- one data point with 3 features
point = np.array([1, 2, 3])
print("1D:", point, "  shape:", point.shape)

# 2D array -- 4 data points, 2 features each
X = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
    [7.0, 8.0]
])
print("2D:\n", X)
print("shape:", X.shape, "  -> 4 points, 2 features")

1D: [1 2 3]   shape: (3,)
2D:
 [[1. 2.]
 [3. 4.]
 [5. 6.]
 [7. 8.]]
shape: (4, 2)   -> 4 points, 2 features


---
## 2. np.zeros and np.zeros_like

Create arrays filled with zeros. We use these to **pre-allocate** space before filling in values.

**Why do we need this in KMeans?**

In KMeans, before computing distances, we need a table to store the results. For example, if we have 4 data points and 2 centroids, we need a 4x2 table where each cell will hold the distance from that point to that centroid. We create this empty table first with `np.zeros`, then fill it in with a loop.

```
np.zeros((rows, cols))   -> create a matrix of given size, all zeros
np.zeros_like(X)         -> create a zero matrix with the SAME shape as X
```

**KMeans usage pattern:**

```python
# Step 1: Create empty distance table
dists = np.zeros((N_points, K))     # one row per point, one col per centroid

# Step 2: Fill in distances (loop over centroids)
for k in range(K):
    dists[:, k] = ...               # fill column k with distances to centroid k

# Similarly for new centroids:
new_centroids = np.zeros_like(centroids)   # same shape as old centroids
for k in range(K):
    new_centroids[k] = ...                  # fill row k with new centroid position
```

In [ ]:
# np.zeros: create a table with specific size, filled with 0
# We want a distance table: 4 points x 2 centroids = 4 rows, 2 columns
dists = np.zeros((4, 2))
print("Empty distance table (4 points x 2 centroids):")
print(dists)
print("Shape:", dists.shape)

In [ ]:
# np.zeros_like: create a zero array with the SAME shape as another array
# Useful when you want a "blank copy" of your centroids to fill with new values
centroids = np.array([[3.0, 4.0],
                      [7.0, 8.0]])
print("Old centroids:")
print(centroids)
print("Shape:", centroids.shape)

new_centroids = np.zeros_like(centroids)
print("\nnp.zeros_like(centroids) -- same shape, all zeros:")
print(new_centroids)
print("Shape:", new_centroids.shape)
print("\nNow we can fill in new_centroids[0] and new_centroids[1] with updated values")

In [ ]:
# Now fill in the table -- imagine we computed some distances
dists[0, 0] = 1.5   # point 0 to centroid 0
dists[0, 1] = 7.2   # point 0 to centroid 1
dists[1, 0] = 5.3   # point 1 to centroid 0
dists[1, 1] = 2.1   # point 1 to centroid 1
# ... and so on

print("After filling in some values:")
print(dists)
print("\nIn KMeans we fill ALL values using a loop (see Section 8)")

---
## 3. Indexing and Slicing

Access specific rows, columns, or elements.

```
X[0]       -> first row (first data point)
X[:, 0]    -> first column (first feature of all points)
X[1:3]     -> rows 1 and 2 (slice)
```

In [18]:
print("X:\n", X)
print()
print("X[0]     -> first point:    ", X[0])
print("X[2]     -> third point:    ", X[2])
print("X[:, 0]  -> all x-coords:  ", X[:, 0])
print("X[:, 1]  -> all y-coords:  ", X[:, 1])
print("X[1:3]   -> rows 1-2:")
print(X[1:3])

X:
 [[1. 2.]
 [3. 4.]
 [5. 6.]
 [7. 8.]]

X[0]     -> first point:     [1. 2.]
X[2]     -> third point:     [5. 6.]
X[:, 0]  -> all x-coords:   [1. 3. 5. 7.]
X[:, 1]  -> all y-coords:   [2. 4. 6. 8.]
X[1:3]   -> rows 1-2:
[[3. 4.]
 [5. 6.]]


---
## 4. The `axis` Parameter

This is the most important concept. Many NumPy functions accept `axis`:

- **`axis=0`** -- operate **down** the rows (collapse rows, keep columns)
- **`axis=1`** -- operate **across** the columns (collapse columns, keep rows)
- **No axis** -- operate on the entire array

```
       col0  col1
row0 [  1     2  ]     axis=0 goes DOWN     (result has shape of one row)
row1 [  3     4  ]     axis=1 goes ACROSS   (result has shape of one column)
row2 [  5     6  ]
```

In [19]:
A = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
])
print("A:\n", A)
print("\nnp.sum(A)          =", np.sum(A),          "  (sum everything)")
print("np.sum(A, axis=0)  =", np.sum(A, axis=0),  "  (sum down each column)")
print("np.sum(A, axis=1)  =", np.sum(A, axis=1),  "  (sum across each row)")

A:
 [[1 2]
 [3 4]
 [5 6]]

np.sum(A)          = 21   (sum everything)
np.sum(A, axis=0)  = [ 9 12]   (sum down each column)
np.sum(A, axis=1)  = [ 3  7 11]   (sum across each row)


In [20]:
# mean with axis -- used to compute new centroids
print("A.mean(axis=0) =", A.mean(axis=0), "  (mean of each column = centroid)")
print("A.mean(axis=1) =", A.mean(axis=1), "  (mean of each row)")

A.mean(axis=0) = [3. 4.]   (mean of each column = centroid)
A.mean(axis=1) = [1.5 3.5 5.5]   (mean of each row)


**Why this matters for KMeans:**

When we compute the new centroid of a cluster, we take the mean of all points in that cluster **along axis=0** (down the rows). Each row is a point, each column is a feature.

```python
cluster_points = np.array([[1, 2],
                           [3, 4],
                           [5, 6]])
new_centroid = cluster_points.mean(axis=0)  # -> [3.0, 4.0]
```

---
## 5. np.argmin and np.argmax

Return the **index** of the min/max value. In KMeans, `np.argmin` assigns each point to its nearest centroid.

```
dists = [5.2, 1.3, 8.0]     (distances to 3 centroids)
np.argmin(dists) = 1         (centroid 1 is closest)
```

In [21]:
# Suppose we have 3 centroids:
#   index 0 -> C0 = (2, 3)
#   index 1 -> C1 = (8, 9)
#   index 2 -> C2 = (5, 1)
# And one point p = (7, 8). Its distance to each centroid:

dists_1point = np.array([5.2, 1.3, 8.0])

print("Centroid labels and distances from point p:")
print(f"  index 0 (C0): distance = {dists_1point[0]}")
print(f"  index 1 (C1): distance = {dists_1point[1]}  <-- smallest")
print(f"  index 2 (C2): distance = {dists_1point[2]}")
print()
print(f"np.argmin({dists_1point}) = {np.argmin(dists_1point)}")
print(f"  -> returns index 1, meaning this point is assigned to C1")

Centroid labels and distances from point p:
  index 0 (C0): distance = 5.2
  index 1 (C1): distance = 1.3  <-- smallest
  index 2 (C2): distance = 8.0

np.argmin([5.2 1.3 8. ]) = 1
  -> returns index 1, meaning this point is assigned to C1


In [22]:
# 2D example: 4 points, 2 centroids (C0 and C1)
# Each row = one point, each column = distance to that centroid
#
#              col 0 (C0)    col 1 (C1)
# row 0:        0.5           7.2       -> min at col 0 -> assign to C0
# row 1:        5.2           1.3       -> min at col 1 -> assign to C1
# row 2:        6.1           2.0       -> min at col 1 -> assign to C1
# row 3:        0.8           9.4       -> min at col 0 -> assign to C0

dist_table = np.array([
    [0.5, 7.2],
    [5.2, 1.3],
    [6.1, 2.0],
    [0.8, 9.4]
])

print("Distance table:")
print("         C0     C1")
for i in range(len(dist_table)):
    print(f"  pt{i}:  {dist_table[i,0]:5.1f}  {dist_table[i,1]:5.1f}")

print()

# np.argmin with axis=1: for EACH ROW, find the column index of the smallest value
labels = np.argmin(dist_table, axis=1)
print("np.argmin(dist_table, axis=1) =", labels)
print()
for i in range(len(labels)):
    print(f"  point {i} -> assigned to C{labels[i]} (column {labels[i]} had the smallest distance)")

Distance table:
         C0     C1
  pt0:    0.5    7.2
  pt1:    5.2    1.3
  pt2:    6.1    2.0
  pt3:    0.8    9.4

np.argmin(dist_table, axis=1) = [0 1 1 0]

  point 0 -> assigned to C0 (column 0 had the smallest distance)
  point 1 -> assigned to C1 (column 1 had the smallest distance)
  point 2 -> assigned to C1 (column 1 had the smallest distance)
  point 3 -> assigned to C0 (column 0 had the smallest distance)


---
## 6. Boolean Indexing (Masking)

Select rows that satisfy a condition. In KMeans, we use this to get all points belonging to a specific cluster.

```
labels = [0, 1, 1, 0]
X[labels == 0]  -> rows where label is 0
X[labels == 1]  -> rows where label is 1
```

In [23]:
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
labels = np.array([0, 1, 1, 0])

# Step 1: create a boolean mask
mask = labels == 0
print("labels:       ", labels)
print("labels == 0:  ", mask, "  (True where label is 0)")
print()

# Step 2: use mask to select rows
cluster_0 = X[mask]
cluster_1 = X[labels == 1]
print("Cluster 0 points:\n", cluster_0)
print("Cluster 1 points:\n", cluster_1)
print()

# Step 3: compute centroid as mean of cluster points
centroid_0 = cluster_0.mean(axis=0)
centroid_1 = cluster_1.mean(axis=0)
print("Centroid 0:", centroid_0, "  = mean of", cluster_0.tolist())
print("Centroid 1:", centroid_1, "  = mean of", cluster_1.tolist())

labels:        [0 1 1 0]
labels == 0:   [ True False False  True]   (True where label is 0)

Cluster 0 points:
 [[1 2]
 [7 8]]
Cluster 1 points:
 [[3 4]
 [5 6]]

Centroid 0: [4. 5.]   = mean of [[1, 2], [7, 8]]
Centroid 1: [4. 5.]   = mean of [[3, 4], [5, 6]]


---
## 7. Broadcasting (Array - Scalar/Vector)

NumPy automatically expands smaller arrays to match larger ones during operations. This is how we compute the distance from **all points** to **one centroid** in a single line.

```
X shape:        (4, 2)     -- 4 points
centroid shape: (2,)       -- 1 point
X - centroid:   (4, 2)     -- NumPy broadcasts centroid to every row
```

In [24]:
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
centroid = np.array([3, 4])

print("X (4 points):")
print(X)
print("\ncentroid:", centroid)
print("\nX - centroid (broadcast):")
print(X - centroid)
print("\nThis subtracted [3,4] from EVERY row automatically.")
print("Row 0: [1,2] - [3,4] = [-2,-2]")
print("Row 1: [3,4] - [3,4] = [ 0, 0]")
print("Row 2: [5,6] - [3,4] = [ 2, 2]")
print("Row 3: [7,8] - [3,4] = [ 4, 4]")

X (4 points):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

centroid: [3 4]

X - centroid (broadcast):
[[-2 -2]
 [ 0  0]
 [ 2  2]
 [ 4  4]]

This subtracted [3,4] from EVERY row automatically.
Row 0: [1,2] - [3,4] = [-2,-2]
Row 1: [3,4] - [3,4] = [ 0, 0]
Row 2: [5,6] - [3,4] = [ 2, 2]
Row 3: [7,8] - [3,4] = [ 4, 4]


---
## 8. np.linalg.norm -- Euclidean Distance

Computes the length (norm) of a vector. With `axis=1`, it computes the norm of each row.

**Single distance:**
$$d(x, c) = \|x - c\| = \sqrt{(x_1-c_1)^2 + (x_2-c_2)^2}$$

**All distances at once (broadcasting + axis=1):**
```python
np.linalg.norm(X - centroid, axis=1)  # distance from every point to one centroid
```

In [25]:
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
centroid = np.array([3, 4])

# Manual calculation for point [1, 2]
diff = X[0] - centroid
print("Manual: [1,2] - [3,4] =", diff)
print("        sqrt((-2)^2 + (-2)^2) = sqrt(8) =", np.sqrt(8))
print("        np.linalg.norm(diff)   =", np.linalg.norm(diff))
print()

# All at once -- this is how KMeans does it
all_dists = np.linalg.norm(X - centroid, axis=1)
print("np.linalg.norm(X - centroid, axis=1):")
print(all_dists)
print("\nBreakdown:")
for i in range(len(X)):
    print(f"  point {X[i]} -> centroid {centroid} = {all_dists[i]:.3f}")

Manual: [1,2] - [3,4] = [-2 -2]
        sqrt((-2)^2 + (-2)^2) = sqrt(8) = 2.8284271247461903
        np.linalg.norm(diff)   = 2.8284271247461903

np.linalg.norm(X - centroid, axis=1):
[2.82842712 0.         2.82842712 5.65685425]

Breakdown:
  point [1 2] -> centroid [3 4] = 2.828
  point [3 4] -> centroid [3 4] = 0.000
  point [5 6] -> centroid [3 4] = 2.828
  point [7 8] -> centroid [3 4] = 5.657


---
## 9. np.allclose -- Convergence Check

Checks if two arrays are element-wise equal within a tolerance. Used to detect when centroids stop moving.

```python
np.allclose(old_centroids, new_centroids)  # True = converged
```

In [26]:
old = np.array([1.0, 2.0])
new_moved = np.array([1.5, 2.3])
new_same = np.array([1.0, 2.0])
new_tiny = np.array([1.0000001, 2.0000001])

print("Centroid moved significantly:")
print(f"  old={old}, new={new_moved}")
print(f"  np.allclose = {np.allclose(old, new_moved)}  -> keep iterating\n")

print("Centroid stayed exactly the same:")
print(f"  old={old}, new={new_same}")
print(f"  np.allclose = {np.allclose(old, new_same)}  -> converged\n")

print("Centroid moved by a tiny amount (floating point):")
print(f"  old={old}, new={new_tiny}")
print(f"  np.allclose = {np.allclose(old, new_tiny)}  -> converged (within tolerance)")

Centroid moved significantly:
  old=[1. 2.], new=[1.5 2.3]
  np.allclose = False  -> keep iterating

Centroid stayed exactly the same:
  old=[1. 2.], new=[1. 2.]
  np.allclose = True  -> converged

Centroid moved by a tiny amount (floating point):
  old=[1. 2.], new=[1.0000001 2.0000001]
  np.allclose = True  -> converged (within tolerance)


---
## 10. np.amax and np.sum -- Purity Score

The purity score function in `week_3.ipynb` uses:

```python
np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)
```

- `np.amax(M, axis=0)` -- max value in each **column** (for each predicted cluster, find the dominant true class count)
- `np.sum(...)` -- sum those maxima
- Divide by total to get purity

In [27]:
# Example contingency matrix:
#            Predicted C0  Predicted C1
# True 0:       3              0
# True 1:       1              3

M = np.array([[3, 0],
              [1, 3]])

print("Contingency matrix:")
print(M)

col_max = np.amax(M, axis=0)  # max in each column
print("\nnp.amax(M, axis=0) =", col_max, "  (dominant class count per cluster)")

purity = np.sum(col_max) / np.sum(M)
print(f"Purity = sum({col_max}) / sum(all) = {np.sum(col_max)} / {np.sum(M)} = {purity:.3f}")

Contingency matrix:
[[3 0]
 [1 3]]

np.amax(M, axis=0) = [3 3]   (dominant class count per cluster)
Purity = sum([3 3]) / sum(all) = 6 / 7 = 0.857


---
## 11. Putting It All Together -- One KMeans Iteration

Here is how all these operations combine in a single iteration of KMeans:

```
1. np.linalg.norm(X - centroid, axis=1)   -> distances (broadcast + norm)
2. np.argmin(dists, axis=1)               -> assign to nearest
3. X[labels == k]                         -> get cluster members (boolean mask)
4. members.mean(axis=0)                   -> new centroid (mean along axis 0)
5. np.allclose(old, new)                  -> check convergence
```

In [28]:
# Full example: 6 points, K=2
X = np.array([[1,2], [1.5,1.8], [5,8], [8,8], [1,0.6], [9,11]])
centroids = np.array([[1.5, 1.8], [5.0, 8.0]])  # initial centroids
K = 2

print("=== Step 1: Compute distances ===")
dists = np.zeros((len(X), K))               # (6, 2) zero matrix
for k in range(K):
    dists[:, k] = np.linalg.norm(X - centroids[k], axis=1)  # broadcast + norm
print("Distance table:\n", dists.round(3))

print("\n=== Step 2: Assign to nearest ===")
labels = np.argmin(dists, axis=1)            # argmin across columns
print("Labels:", labels)

print("\n=== Step 3: Get cluster members ===")
for k in range(K):
    members = X[labels == k]                  # boolean indexing
    print(f"  Cluster {k}: {members.tolist()}")

print("\n=== Step 4: Compute new centroids ===")
new_centroids = np.zeros_like(centroids)      # zeros_like
for k in range(K):
    new_centroids[k] = X[labels == k].mean(axis=0)  # mean along axis 0
    print(f"  C{k+1}: {centroids[k]} -> {new_centroids[k]}")

print("\n=== Step 5: Check convergence ===")
print(f"  np.allclose = {np.allclose(centroids, new_centroids)}")
print(f"  {'Converged' if np.allclose(centroids, new_centroids) else 'Not converged -- repeat'}")

=== Step 1: Compute distances ===
Distance table:
 [[ 0.539  7.211]
 [ 0.     7.12 ]
 [ 7.12   0.   ]
 [ 8.983  3.   ]
 [ 1.3    8.412]
 [11.87   5.   ]]

=== Step 2: Assign to nearest ===
Labels: [0 0 1 1 0 1]

=== Step 3: Get cluster members ===
  Cluster 0: [[1.0, 2.0], [1.5, 1.8], [1.0, 0.6]]
  Cluster 1: [[5.0, 8.0], [8.0, 8.0], [9.0, 11.0]]

=== Step 4: Compute new centroids ===
  C1: [1.5 1.8] -> [1.16666667 1.46666667]
  C2: [5. 8.] -> [7.33333333 9.        ]

=== Step 5: Check convergence ===
  np.allclose = False
  Not converged -- repeat
